Trade util

In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import backtrader as bt
from datetime import datetime

In [11]:
from tqsdk.tafunc import ma, ema, abs, std, hhv, llv, count, time_to_datetime, barlast

在使用天勤量化之前，默认您已经知晓并同意以下免责条款，如果不同意请立即停止使用：https://www.shinnytech.com/blog/disclaimer/


In [2]:
def gen_stop_loss_long(data,para):
#     base = data[['close','zd_strength','ma_body']].copy()
    #     大阳
#     大阳 = base.zd_strength >= para['K线']['大阴']
#     bsp_label = base.bsp_label >= 2 
#     bsp_relative_yc = base.relative >=3
#     fl = base.v1_strength>para['volume']['fl_v1_strength']
#     combination = 大阳 + bsp_label + bsp_relative_yc + fl
#     condition = combination & (base.pct_chg >= 0)
    base = data.copy()
    
#     下面的止损太快，没法抓住趋势
#     condition = (base.zd_strength >= 0)
#     temp = base.close*condition - base.ma_body * condition * para['stop_loss']['long']['buffer_ratio']

# 下面的止损在开仓后 容易被摔出
#     base = data.copy()
#     temp = base.long*base.ma3 

#     中阳 = (base.zd_strength >= para['K线']['中阳']) & (base.zd_strength < para['K线']['大阳'])
#     大阳 = base.zd_strength >= para['K线']['大阳']
#     temp = (大阳*base.close - base.ma_body * 大阳 * 3 ) + (中阳*base.open - 中阳*base.ma_body)
    
#     找到大于2倍ma_boday 的大阳 做标准化, 以阳线开盘-ma_body 作为止损线
    大阳 = base.zd_strength >= para['K线']['大阳']
    中阳 = (base.zd_strength >= para['K线']['中阳']) & (大阳==False)
    temp = (大阳*base.close - base.ma_body * 大阳 * 3 ) + (中阳*base.open - 中阳*base.ma_body)
    return temp.replace(0, np.nan).ffill()


In [3]:
def gen_stop_loss_short(data,para):
    base = data.copy()
    #     大阴
    大阴 = base.zd_strength <= para['K线']['大阴']
    bsp= base.bsp_label >= 2

In [4]:
# 多头震荡 做空
def gen_long_signal(data,para):
    long1 = long_pattern1(data,para)
    long2 = long_pattern2(data,para)
    long = long1 + long2
    return long
#     return long1,long2

In [5]:
# 大多中拢震荡底部启动多头趋势
def long_pattern1(data,para):
    base1 = data.copy()
    ma_pattern = (base1.ma_category_l == 4) & (base1.kl_ma35 == 1)
    peak_pattern = (base1.peak2_bias >= 0) & (base1.peak2_bias < 5)
    价格合适 = (abs(base1.bias1) <4)

    # 单面上涨，中间没有波动
    up_side =  (base1.v_pos1 == base1.v_pos3) & (base1.p_pos1>0)
    tupo_momentum = (base1.mv_stage == 3) & (base1.zd_strength >= 2) 
    大多中拢震荡底部启动= ma_pattern & tupo_momentum & peak_pattern &价格合适 & up_side
    return 大多中拢震荡底部启动

In [6]:
# 大6(多反转)中空震荡底部启动反弹做多趋势
def long_pattern2(data,para):
    base = data.copy()
    ma_pattern = (base.ma_category_l == 6) & (base.ma_category_m == 1) 
    mv_pattern = (base.mv_stage ==1)
    v_pattern = (base.volume/base.volume.shift(1) > 1.1)
    
    peak_pattern= (base.peak1_bias < 5) & (base.peak2_bias < 4) & (base.peak3_bias < 3)
    bias_pattern = (base.bias1<5) & (base.bias2<5) & (base.bias3<5)

    两阳加速 = (base.body_strength/base.body_strength.shift(1) >1.5) & (base.body_strength > 1.5) & (base.body_strength.shift(1) > 0.75)
    大6中空超跌 = peak_pattern & ma_pattern  & v_pattern & 两阳加速 & mv_pattern 
    return 大6中空超跌

In [9]:
# 大1(空)中1(空)小(kl)  反抽后续跌
def short_pattern1(data,para):
    base = data.copy()
    ma_pattern = (base.ma_category_l == 1) & (base.ma_category_m == 1) & (base.kl_ma13 == 1) 
    mv_pattern = (base.mv_stage >= 2) #放量
    peak_pattern = (base.p_prev1_2 >= base.peak1)
    k_pattern = (base.pct_chg < -base.ma_pct) & (base.k_ups/base.ma_ups >=2)
    bias_pattern = (abs(base.bias1)< 1.5 )
    return ma_pattern & mv_pattern & peak_pattern & k_pattern & bias_pattern

In [ ]:
# def short_pattern2(data,para):
#     base = data.copy()
    

In [ ]:
# 大4中4小4回踩
# def long_pattern3(data,para):
#     base = data.copy()

In [10]:
def gen_short_signal(data,para):
    long1 = pattern1(data,para)
    long2 = pattern2(data,para)
    long = long1+long2
    return long